In [1]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

connected to port: 60644


In [2]:
treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    matrix_type="coverage",
)

In [ ]:
X = corpus.feature_matrix
print("Feature matrix shape:", X.shape)

In [ ]:
X

In [ ]:
import skfuzzy as fuzz
import numpy as np
Xd = np.asarray(X, dtype=float)
n_clusters = 10 # obtained with dunn and db index
m = 2 
# scikit-fuzzy expects (features, samples), so transpose
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    Xd.T, c=n_clusters, m=m, error=1e-5, maxiter=1000, init=None, seed=42
)
labels = u.argmax(axis=0)

In [ ]:
membership = u.T
membership

In [ ]:
fpc

In [ ]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)

In [ ]:
fig = tod.plotting.fuzzy_cluster_scatter_plot(corpus, dim_red, membership)

In [ ]:
fig.write_html("fuzzy_cluster_scatter_plot.html")

In [ ]:
def fuzzy_multi_cluster_lexunits(corpus: Corpus, membership: np.ndarray, threshold: float = 0.3):
    """
    Return list of (lexunit, memberships_dict) for lexical units having membership
    >= threshold in at least 2 clusters.
    membership shape: (n_samples, n_clusters)
    """
    result = []
    for i, row in enumerate(membership):
        passed = {c: float(m) for c, m in enumerate(row) if m >= threshold}
        if len(passed) >= 2:
            result.append((corpus.idx2lexunit(i), passed))
    return result

def fuzzy_pair_overlaps(corpus: Corpus, membership: np.ndarray, threshold: float = 0.3):
    """
    Return dict mapping (cluster_a, cluster_b) -> list of lexunits that have membership
    >= threshold in both clusters.
    """
    n_clusters = membership.shape[1]
    overlaps = {}
    for i, row in enumerate(membership):
        active = [c for c, m in enumerate(row) if m >= threshold]
        if len(active) >= 2:
            lex = corpus.idx2lexunit(i)
            for a in range(len(active)):
                for b in range(a + 1, len(active)):
                    pair = (active[a], active[b])
                    overlaps.setdefault(pair, []).append(lex)
    return overlaps

def fuzzy_ambiguous_lexunits(corpus: Corpus, membership: np.ndarray, top_k: int = 2, max_gap: float = 0.15):
    """
    Return lexical units whose top_k memberships are close (difference between
    rank 1 and rank top_k <= max_gap).
    """
    result = []
    for i, row in enumerate(membership):
        order = np.argsort(row)[::-1]
        top_vals = row[order][:top_k]
        if len(top_vals) == top_k and (top_vals[0] - top_vals[-1]) <= max_gap:
            result.append((corpus.idx2lexunit(i), [(int(c), float(row[c])) for c in order[:top_k]]))
    return result

In [ ]:
# Get units with membership >= 0.35 in at least two clusters
multi = fuzzy_multi_cluster_lexunits(corpus, membership, threshold=0.35)
print(f"{len(multi)} multi-cluster lexical units")
print(multi[:10])

# Pairs overlap
pairs = fuzzy_pair_overlaps(corpus, membership, threshold=0.35)
for pair, lexunits in list(pairs.items())[:5]:
    print("Pair", pair, "count", len(lexunits))

# Ambiguous (close top 2 memberships)
amb = fuzzy_ambiguous_lexunits(corpus, membership, top_k=2, max_gap=0.1)
print(f"{len(amb)} ambiguous lexical units")
print(amb[:10])

In [ ]:
print("Membership per row variance (first 10):",
      [float(np.var(r)) for r in membership[:10]])
print("All membership rows identical? ",
      np.allclose(membership, membership[0]))
print("Row sums (should be 1):", membership.sum(axis=1)[:5])
print("Cluster center pairwise max diff:",
      np.max([np.max(np.abs(c1 - c2)) for i,c1 in enumerate(cntr) for j,c2 in enumerate(cntr) if i<j]))
print("Per-feature std (first 10):", np.std(Xd, axis=0)[:10])
print("Zero-variance feature count:", np.sum(np.std(Xd, axis=0)==0))

In [ ]:
best = None
for c in range(2, 14):
    _, u, _, _, _, _, fpc = fuzz.cluster.cmeans(Xd.T, c=c, m=2.0, error=1e-5, maxiter=1000, seed=42)
    if best is None or fpc > best[0]:
        best = (fpc, c, u)
print(f"Best c by FPC: {best[1]} (FPC={best[0]:.4f})")

In [ ]:
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

# Dense and safe
Xd = X.toarray() if hasattr(X, "toarray") else np.asarray(X, dtype=float)
Xd = np.nan_to_num(Xd, copy=False)

# 1) Drop lowest-variance 20% of features (keeps structure, removes near-constant cols)
col_std = Xd.std(axis=0)
keep = col_std > np.percentile(col_std, 20)
Xd1 = Xd[:, keep]

# 2) Row L2-normalize (approximates cosine distance)
Xd1 = normalize(Xd1, axis=1)

# 3) Scale columns (avoid mean-centering to keep sparsity semantics)
Xd2 = StandardScaler(with_mean=False).fit_transform(Xd1)

# 4) Dimensionality reduction (denoise + avoid collinearity)
n_comp = min(100, Xd2.shape[1]-1) if Xd2.shape[1] > 50 else max(2, Xd2.shape[1])
if n_comp < Xd2.shape[1]:
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    Xf = svd.fit_transform(Xd2)
else:
    Xf = Xd2

print("Preprocessed shape:", Xf.shape)

# 5) Choose clusters
n_clusters = 8

# 6) KMeans to build an initial membership matrix (one-hot)
km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(Xf)
labels0 = km.labels_                            # length = n_samples
u0_init = np.zeros((n_clusters, Xf.shape[0]))   # (c, n_samples)
u0_init[labels0, np.arange(Xf.shape[0])] = 1.0

# 7) Fuzzy c-means (expects features x samples)
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    Xf.T, c=n_clusters, m=2.0, error=1e-5, maxiter=1000, init=u0_init, seed=42
)

membership = u.T             # shape: (n_words, n_clusters)
labels = membership.argmax(axis=1)

print("FPC:", fpc, " avg membership row var:", float(membership.var(axis=1).mean()))
print("Cluster sizes:", np.bincount(labels))

In [3]:
clustering = tod.clustering.SparseKMeans(corpus=corpus, k=10, top_n_features=10, cluster_defining_features=True)
centroids = clustering.get_centroids(corpus)

Cluster 0 - Top 10 defining features:
  1. node:X:child:rel_shallow=det                       importance: 0.113808
  2. node:X:child:upos=DET                              importance: 0.112453
  3. node:X:own:Gender=Masc                             importance: 0.103981
  4. node:X:prev:upos=DET                               importance: 0.083910
  5. node:X:child:PronType=Art                          importance: 0.083604
  6. node:X:child:rel_shallow=case                      importance: 0.078241
  7. node:X:parent:position=before                      importance: 0.063118
  8. node:X:prev:PronType=Art                           importance: 0.062866
  9. node:X:child:upos=ADP                              importance: 0.058849
 10. node:X:child:Definite=Def                          importance: 0.052577

Cluster 1 - Top 10 defining features:
  1. node:X:own:Gender=Fem                              importance: 0.161387
  2. node:X:child:rel_shallow=det                       importance: 0.112800

In [4]:
centroids.shape

(10, 1976)

In [7]:
import numpy as np
import skfuzzy as fuzz

# Data as dense float, shape (n_samples, n_features)
X = corpus.feature_matrix
Xd = X.toarray() if hasattr(X, "toarray") else np.asarray(X, dtype=float)

# Centroids from SparseKMeans: shape (c, n_features)
centroids = np.asarray(centroids, dtype=float)

m = 1.5
# Use cmeans_predict with fixed centers
u, u0, d, jm, p, fpc = fuzz.cluster.cmeans_predict(
    Xd.T, centroids, m=m, error=1e-5, maxiter=1000, seed=42
)
labels = u.argmax(axis=0)


In [ ]:
import numpy as np
k_eff = np.exp((-membership * np.log(membership + 1e-12)).sum(axis=1))
print("Mean effective clusters:", float(k_eff.mean()))

In [8]:
fpc

0.6213022022089072

In [ ]:
import numpy as np
c = u.shape[0]
print("1/c baseline:", 1.0/c)
print("avg max membership:", float(u.T.max(axis=1).mean()))


In [ ]:
# Average membership entropy (lower => crisper)
import numpy as np
eps = 1e-12
entropy = (-membership * np.log(membership + eps)).sum(axis=1)
print("Mean entropy:", float(entropy.mean()), "max possible:", np.log(membership.shape[1]))

In [ ]:
membership = u.T
membership

In [9]:
import numpy as np
import pandas as pd

# Ensure membership is (n_samples, n_clusters)
membership = u.T  # if not already set

threshold = 0.1
hard_labels = membership.argmax(axis=1)  # hard (argmax) cluster per sample

mask = membership >= threshold
multi_rows = np.where(mask.sum(axis=1) >= 2)[0]  # row indices in original X

rows = []
for i in multi_rows:
    clusters = np.where(mask[i])[0]
    rows.append({
        "row_idx": int(i),                                   # row in X
        "lexunit": corpus.idx2lexunit(i),                    # optional: your item id
        "primary_cluster": int(hard_labels[i]),              # argmax cluster
        "clusters_above_threshold": [(int(c), float(membership[i, c])) for c in clusters]
    })

df_multi = pd.DataFrame(rows).sort_values("row_idx")
print(f"{len(df_multi)} items are in >=2 clusters (threshold={threshold}).")
display(df_multi)

930 items are in >=2 clusters (threshold=0.1).


,row_idx,lexunit,primary_cluster,clusters_above_threshold
0,0,"($, NOUN)",3,"[(0, 0.11128177021329737), (1, 0.1074284389927..."
1,1,"(%, NOUN)",3,"[(0, 0.12349713672603001), (1, 0.1202365057415..."
2,2,"(&, CCONJ)",4,"[(4, 0.16615745423851977), (5, 0.1138812698801..."
3,3,"(/, ADP)",4,"[(4, 0.20808343100090668), (6, 0.1107797861666..."
4,4,"(/, CCONJ)",4,"[(4, 0.23392440283354143), (6, 0.1086485350205..."
...,...,...,...,...
925,2927,"(être, AUX)",4,"[(4, 0.288747466407756), (8, 0.159454022279456..."
926,2929,"(île, NOUN)",1,"[(0, 0.12375985115280737), (1, 0.8171597909439..."
927,2930,"(œil, NOUN)",0,"[(0, 0.6764927443513338), (1, 0.14604965395020..."
928,2931,"(œuf, NOUN)",0,"[(0, 0.5712598007150288), (1, 0.10902782556952..."


In [10]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)
fig = tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering)
fig

/opt/homebrew/lib/python3.11/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [11]:
fig.write_html("sparse_kmeans_gsd.html")